# Weeks 2–3 – Mortgage Rate Enrichment

**Status:** Handbook deliverable complete  
**MLS coverage:** January 2024 through September 2026  
**Economic series:** FRED `MORTGAGE30US`

This notebook documents the reproducible enrichment of both private Residential datasets with the national 30-year fixed mortgage rate.

## Objective and handbook method

The handbook requires:

1. Fetch `MORTGAGE30US` directly from FRED without an API key.
2. Convert weekly observations to calendar-month averages.
3. Create `year_month` from `ListingContractDate` for listings.
4. Create `year_month` from `CloseDate` for sold records.
5. Left-merge the monthly rate into both datasets.
6. Confirm the merge preserves row counts and leaves zero null rates.
7. Save both enriched datasets as private CSV files.

## Confidentiality rule

The FRED series is public. The MLS inputs and enriched outputs remain private under `data/processed/`. Notebook output is limited to public monthly rates and aggregate validation counts.

## 1. Environment and reusable imports

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.weeks2_3_mortgage import (
    EXPECTED_MONTHS,
    EXPECTED_ROWS,
    OUTPUT_FILES,
    enrich_dataset,
    fetch_monthly_mortgage_rates,
)

## 2. Fetch and aggregate the public FRED series

In [2]:
mortgage_monthly = fetch_monthly_mortgage_rates()

required_mortgage_rates = mortgage_monthly.loc[
    mortgage_monthly["year_month"].isin(EXPECTED_MONTHS)
].reset_index(drop=True)

required_mortgage_rates

FRED MORTGAGE30US
Required monthly coverage: 2024-01 through 2026-09
Required months covered: 33
Rate range: 6.048% to 7.060%


,year_month,rate_30yr_fixed,weekly_observations
0,2024-01,6.6425,4
1,2024-02,6.7760,5
2,2024-03,6.8200,4
3,2024-04,6.9925,4
4,2024-05,7.0600,5
5,2024-06,6.9175,4
6,2024-07,6.8475,4
7,2024-08,6.5000,5
8,2024-09,6.1800,4
9,2024-10,6.4280,5


### FRED finding

All 33 required months are available. Monthly average rates across the project window range from 6.048% to 7.060%. Each value is the arithmetic mean of the available weekly Thursday observations in that calendar month.

## 3. Enrich and save both private datasets

This cell intentionally writes the two private enriched CSV files. The reusable function validates date completeness, row counts, many-to-one join cardinality, month coverage, and null mortgage rates before reporting success.

In [3]:
listing_output = enrich_dataset(
    dataset="listings",
    mortgage_monthly=mortgage_monthly,
)
sold_output = enrich_dataset(
    dataset="sold",
    mortgage_monthly=mortgage_monthly,
)

print(listing_output.relative_to(PROJECT_ROOT))
print(sold_output.relative_to(PROJECT_ROOT))


LISTINGS ENRICHMENT
Rows before merge: 655,943
Rows after merge: 655,943
Null mortgage rates: 0
Source months: 33
Saved privately: data/processed/combined_listings_residential_with_mortgage_rates.csv

SOLD ENRICHMENT
Rows before merge: 495,334
Rows after merge: 495,334
Null mortgage rates: 0
Source months: 33
Saved privately: data/processed/combined_sold_residential_with_mortgage_rates.csv
data/processed/combined_listings_residential_with_mortgage_rates.csv
data/processed/combined_sold_residential_with_mortgage_rates.csv


## 4. Independent aggregate validation

Only the join key and mortgage-rate columns are loaded for this check. No MLS record values are displayed.

In [4]:
validation_rows = []

for dataset, path in OUTPUT_FILES.items():
    check = pd.read_csv(
        path,
        usecols=["year_month", "rate_30yr_fixed"],
    )
    validation_rows.append(
        {
            "dataset": dataset,
            "rows": len(check),
            "expected_rows": EXPECTED_ROWS[dataset],
            "source_months": check["year_month"].nunique(),
            "null_mortgage_rates": check["rate_30yr_fixed"].isna().sum(),
            "maximum_rates_per_month": (
                check.groupby("year_month")["rate_30yr_fixed"]
                .nunique()
                .max()
            ),
        }
    )

enrichment_validation = pd.DataFrame(validation_rows)
enrichment_validation

,dataset,rows,expected_rows,source_months,null_mortgage_rates,maximum_rates_per_month
0,listings,655943,655943,33,0,1
1,sold,495334,495334,33,0,1


## Findings

- Listings retained all 655,943 rows and now contain 77 columns.
- Sold retained all 495,334 rows and now contain 88 columns.
- Both enriched datasets contain 33 `year_month` values.
- Both have zero null `rate_30yr_fixed` values.
- Every month maps to exactly one mortgage-rate value.
- The enriched CSVs remain ignored by Git and are not uploaded.

## Decisions and assumptions

- The monthly rate is an average of weekly FRED observations, exactly as directed by the handbook.
- Listings are aligned to when the listing agreement became effective.
- Sold records are aligned to the closing month.
- A many-to-one merge prevents duplicate economic rows from multiplying MLS records.
- Missing or invalid join dates are blocking errors because they would prevent complete rate enrichment.
- The national mortgage rate is contextual economic data; it does not represent an individual borrower's actual loan rate.

## Completion checklist

- [x] Fetch `MORTGAGE30US` directly from FRED
- [x] Resample weekly observations to monthly averages
- [x] Create listings key from `ListingContractDate`
- [x] Create sold key from `CloseDate`
- [x] Left-merge onto both datasets
- [x] Preserve both row counts
- [x] Confirm zero null mortgage rates
- [x] Save both enriched datasets privately
- [x] Document the complete workflow in a reusable Python script

**Next handbook phase:** Weeks 4–5 – Data Cleaning and Preparation

## References

- Reusable enrichment code: `src/weeks2_3_mortgage.py`
- Validation and EDA notebook: `notebooks/02_03_data_validation_eda.ipynb`
- Final phase summary: `reports/weeks2_3_final_report.md`
- Chronological notes: `docs/project_log.md`